# Thí nghiệm rò rỉ dữ liệu – Thành viên A

So sánh **chia ngẫu nhiên theo ảnh** với **chia theo tuyến đường (`street_id`)**, cùng một cấu hình YOLOv8n.

Mỗi mô hình được chấm trên 2 tập:
- `val` của chính nó;
- `test` chung = các tuyến đường **chưa từng thấy** (lấy từ `data/splits/val.json`, không đụng tới tập test thật).

**Trước khi chạy:** Settings bên phải → Accelerator = **GPU T4 x2** (hoặc P100), Internet = **On**; Add Input → dataset `za-traffic-2020`.

**Chạy thử trước:** để `SMOKE_TEST = True`, Run All. Chạy ổn thì đổi thành `False`, rồi bấm **Save Version → Save & Run All (Commit)** để Kaggle chạy nền.

In [ ]:
# ==== CẤU HÌNH (chỉ sửa ô này) ====
REPO_URL   = "https://github.com/<ten-tai-khoan>/deeplearning-project-sign-detection.git"   # sửa thành repo nhóm
SMOKE_TEST = True      # True: chạy thử 1 epoch trên 10% dữ liệu (~vài phút). False: chạy thật
EPOCHS     = 50
IMGSZ      = 1280
BATCH      = 16        # nếu báo hết bộ nhớ GPU (CUDA out of memory) thì giảm còn 8
USE_WANDB  = True      # cần secret WANDB_API_KEY (Add-ons → Secrets)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics pycocotools wandb

In [ ]:
# Lấy code và nhãn từ GitHub. Repo private: thêm secret GH_TOKEN (Personal access token, quyền đọc repo)
import os, subprocess
os.chdir("/kaggle/working")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GH_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # repo public thì không cần token
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", url, "repo"], check=True)
os.chdir("/kaggle/working/repo")
!ls scripts data/leak

In [ ]:
# Tự tìm thư mục ảnh của bộ dữ liệu trong /kaggle/input
import glob
js = glob.glob("/kaggle/input/**/train_traffic_sign_dataset.json", recursive=True)
assert js, "Không thấy dataset. Kiểm tra đã Add Input za-traffic-2020 chưa."
IMG_DIR = os.path.join(os.path.dirname(js[0]), "images")
print("Ảnh:", IMG_DIR, "-", len(os.listdir(IMG_DIR)), "file")

In [ ]:
# Chuyển nhãn sang YOLO cho 2 cách chia (ảnh là symlink tới /kaggle/input, không tốn dung lượng)
for v in ["random", "street"]:
    !python scripts/coco_to_yolo.py --splits data/leak/{v} --images {IMG_DIR} --out /kaggle/working/yolo_leak/{v}
!cat data/leak/leak_stats.txt

In [ ]:
# Weights & Biases (đề yêu cầu chia sẻ log huấn luyện với giảng viên)
if USE_WANDB:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    !yolo settings wandb=True
else:
    !yolo settings wandb=False

In [ ]:
# Train 2 mô hình với CÙNG cấu hình, chỉ khác cách chia dữ liệu
from ultralytics import YOLO
extra = dict(epochs=1, fraction=0.1) if SMOKE_TEST else dict(epochs=EPOCHS, patience=15)
best = {}
for v in ["random", "street"]:
    model = YOLO("yolov8n.pt")
    model.train(data=f"/kaggle/working/yolo_leak/{v}/data.yaml", imgsz=IMGSZ, batch=BATCH,
                      seed=0, deterministic=True, device=0, workers=4,
                      project="leakage", name=v, exist_ok=True, **extra)
    best[v] = str(model.trainer.save_dir / "weights" / "best.pt")
    print(v, "->", best[v])

In [ ]:
# Xuất dự đoán và chấm bằng script chung của nhóm: mỗi mô hình × (val của nó, tuyến đường chưa thấy)
for v in ["random", "street"]:
    y = f"/kaggle/working/yolo_leak/{v}"
    for split in ["val", "test"]:
        name = f"leak_{v}_{split}"
        !python scripts/export_yolo_preds.py --weights {best[v]} --gt data/leak/{v}/{split}.json --images {y}/images/{split} --class-map {y}/class_map.json --imgsz {IMGSZ} --out preds/{name}.json
        !python scripts/evaluate.py --gt data/leak/{v}/{split}.json --pred preds/{name}.json --name {name} --out reports/eval_leak > /dev/null

In [ ]:
# Bảng kết quả cho báo cáo
import json
rows = []
for v in ["random", "street"]:
    a = json.load(open(f"reports/eval_leak/leak_{v}_val.json"))
    b = json.load(open(f"reports/eval_leak/leak_{v}_test.json"))
    rows.append((v, a["mAP50"], b["mAP50"], a["mAP50_95"], b["mAP50_95"]))
print(f"{'Cách chia':<10} {'mAP50 val':>10} {'mAP50 mới':>10} {'chênh':>7}   {'mAP50-95 val':>12} {'mAP50-95 mới':>12}")
for v, a50, b50, a, b in rows:
    print(f"{v:<10} {a50*100:>10.1f} {b50*100:>10.1f} {(a50-b50)*100:>+7.1f}   {a*100:>12.1f} {b*100:>12.1f}")
print("\n'mới' = các tuyến đường mô hình chưa từng thấy. Chênh lệch lớn ở 'random' = mức rò rỉ.")

In [ ]:
# Gói kết quả để tải về (tab Output sau khi Save Version chạy xong)
!mkdir -p /kaggle/working/out && cp -r preds reports/eval_leak /kaggle/working/out/
for v in ["random", "street"]:
    !mkdir -p /kaggle/working/out/{v} && cp {best[v]} leakage/{v}/results.csv /kaggle/working/out/{v}/ 2>/dev/null; cp leakage/{v}/*.png /kaggle/working/out/{v}/ 2>/dev/null
!cd /kaggle/working && zip -qr leak_outputs.zip out && ls -lh leak_outputs.zip